# Building inventory — two Altair charts

Data: https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv

In [ ]:
import altair as alt
import pandas as pd

alt.data_transformers.disable_max_rows()

DATA_URL = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv"
df = pd.read_csv(DATA_URL)
df.shape

In [ ]:
congress = (
    df.loc[df["Congress Dist"] > 0]
    .assign(
        year_constructed=lambda d: d["Year Constructed"].where(d["Year Constructed"] > 0)
    )
    .groupby("Congress Dist", as_index=False)
    .agg(
        total_sqft=("Square Footage", "sum"),
        mean_year_constructed=("year_constructed", "mean"),
        n=("Square Footage", "count"),
    )
)
congress["Congress Dist"] = congress["Congress Dist"].astype(str)
congress.head()

In [ ]:
top_usage = df["Usage Description"].value_counts().head(10).index.tolist()
usage_subset = df[df["Usage Description"].isin(top_usage)].copy()
usage_subset = usage_subset.loc[usage_subset["Year Constructed"] > 0, ["Year Constructed", "Usage Description"]]
len(usage_subset)

## Visualization 1 — Total square footage by congressional district

Total building square footage in the Illinois state inventory, summed for each U.S. congressional district. Each row is one district; bar length is total Square Footage for all buildings in that district.

Encodings: x is quantitative (sum of square footage). y is nominal (district id), sorted by x so the largest totals appear first. Color is quantitative mean Year Constructed by district, on a sequential viridis scale (purple for older averages, yellow-green for newer). Transformations: drop rows with Congress Dist equal to zero; aggregate to one row per district with sum of Square Footage, count of buildings, and mean Year Constructed where construction year was greater than zero.

In [ ]:
chart_congress = (
    alt.Chart(congress)
    .mark_bar()
    .encode(
        x=alt.X("total_sqft:Q", title="Total square footage (sum)"),
        y=alt.Y("Congress Dist:N", sort="-x", title="Congressional district"),
        tooltip=[
            alt.Tooltip("Congress Dist:N", title="District"),
            alt.Tooltip("total_sqft:Q", format=",.0f", title="Total sq ft"),
            alt.Tooltip("mean_year_constructed:Q", format=".1f", title="Mean year constructed"),
            alt.Tooltip("n:Q", title="Buildings"),
        ],
        color=alt.Color(
            "mean_year_constructed:Q",
            scale=alt.Scale(scheme="viridis"),
            title="Mean year constructed",
        ),
    )
    .properties(
        width=550,
        height=400,
        title="State inventory footprint by congressional district",
    )
)

chart_congress

## Visualization 2 — Construction year and usage mix with brushing

Top: histogram of Year Constructed (binned). Bottom: counts of buildings by Usage Description for rows whose construction year falls in the brushed interval. The analysis uses the ten most common usage labels, drops Year Constructed equal to zero, and keeps only year and usage for the chart data.

Encodings: histogram uses binned quantitative years on x and count on y. Bar chart uses nominal usage on y, count on x, and one hue per usage (legend hidden; labels on the axis). Transformations: restrict to those ten usage values, remove zero construction years, two columns retained for export size. Interactivity: an x-interval brush on the histogram filters the bar chart so the usage mix updates when you drag a different span of construction years. The vertical layout shares the x domain between panels.

In [ ]:
brush = alt.selection_interval(encodings=["x"], empty="all")

hist = (
    alt.Chart(usage_subset)
    .mark_bar()
    .encode(
        x=alt.X("Year Constructed:Q", bin=alt.Bin(maxbins=35), title="Year constructed"),
        y=alt.Y("count():Q", title="Number of buildings"),
        color=alt.value("#4c78a8"),
    )
    .properties(
        width=550,
        height=180,
        title="Brush the histogram to filter usage counts below",
    )
    .add_params(brush)
)

bars = (
    alt.Chart(usage_subset)
    .mark_bar()
    .encode(
        y=alt.Y("Usage Description:N", sort="-x", title="Usage"),
        x=alt.X("count():Q", title="Buildings in selection"),
        color=alt.Color("Usage Description:N", legend=None),
    )
    .transform_filter(brush)
    .properties(width=550, height=280)
)

chart_brush_usage = alt.vconcat(hist, bars).resolve_axis(x="shared")

chart_brush_usage

## Export

Writes Vega-Lite JSON and standalone HTML under assets/.

In [ ]:
from pathlib import Path

assets = Path("assets")
assets.mkdir(exist_ok=True)

chart_congress.save(str(assets / "viz_congress.json"))
chart_brush_usage.save(str(assets / "viz_brush_usage.json"))

chart_congress.save(str(assets / "viz_congress.html"))
chart_brush_usage.save(str(assets / "viz_brush_usage.html"))

sorted(p.name for p in assets.glob("viz_*"))